# 17.6 深度 Q 网络 / Deep Q-Network (DQN)

**中文**：前面的 Q-learning 用一张**表**存 $Q(s,a)$。但状态一多就崩——Atari 游戏的状态是整屏像素($256^{100000}$ 种)，围棋有 $10^{170}$ 个局面，表根本存不下、也永远填不满。解法:用**神经网络 $Q(s,a;\theta)$ 近似 Q 表**。这就是 **DQN(Mnih et al., DeepMind 2015)**——它让 AI **仅凭像素**就在几十款 Atari 游戏上达到人类水平，**引爆了整个深度强化学习时代**。本节从零实现 DQN 平衡一根杆子(CartPole)。
**English**: Q-learning stored $Q(s,a)$ in a **table**. But that breaks with many states — an Atari frame is a full screen of pixels ($256^{100000}$ possibilities), Go has $10^{170}$ positions; a table can't be stored or ever filled. The fix: **approximate the Q-table with a neural network $Q(s,a;\theta)$**. This is **DQN (Mnih et al., DeepMind 2015)** — it let AI reach human level on dozens of Atari games **from pixels alone**, **igniting the entire deep-RL era**. We implement DQN from scratch to balance a pole (CartPole).

---

**中文**：把 Q 表换成神经网络听起来简单，但**直接这么做会训练崩溃**。因为 RL 违反了监督学习的两大前提:
**English**: Swapping the Q-table for a network sounds simple, but **doing it naively makes training collapse**. Because RL violates two assumptions of supervised learning:
1. **数据不独立(相关性)**:连续几步的状态高度相关(一帧接一帧)，不像监督学习那样 i.i.d.,梯度更新互相"打架"。
   **Correlated data**: consecutive states are highly correlated (frame after frame), not i.i.d. like supervised learning, so gradient updates fight each other.
2. **目标在动(非平稳)**:TD 目标 $r+\gamma\max Q(s';\theta)$ 用的是**正在被更新的同一个网络** $\theta$——你追的目标随你自己一起移动，像追自己的影子,极易发散。
   **Moving target**: the TD target $r+\gamma\max Q(s';\theta)$ uses the **same network $\theta$ being updated** — the target moves as you chase it, like chasing your own shadow, easily diverging.

**中文**：DQN 用**两个关键技巧**解决:
**English**: DQN solves both with **two key tricks**:
- **经验回放(Experience Replay)**:把每步的转移 $(s,a,r,s')$ 存进一个大**回放池**,训练时**随机采样**小批量。这打破了数据相关性(随机采样≈i.i.d.),还能**反复复用**每条经验(off-policy 才允许这么做!)。
  **Experience Replay**: store each transition $(s,a,r,s')$ in a large **replay buffer**; train on **random minibatches**. This breaks correlation (random sampling ≈ i.i.d.) and **reuses** each experience many times (only off-policy allows this!).
- **目标网络(Target Network)**:另存一份**冻结的**网络 $\theta^-$ 专门算 TD 目标,每隔若干步才把 $\theta$ 复制过去。目标"暂时不动",训练就稳定了。
  **Target Network**: keep a separate **frozen** copy $\theta^-$ just for computing TD targets, syncing $\theta\to\theta^-$ every few steps. A momentarily fixed target stabilizes training.

$$L(\theta)=\mathbb E_{(s,a,r,s')\sim \text{replay}}\Big[\big(\underbrace{r+\gamma\max_{a'}Q(s',a';\theta^-)}_{\text{目标(冻结网络)}}-Q(s,a;\theta)\big)^2\Big]$$

> 💡 **面试速查 / Interview cheat-sheet（★★★ 深度 RL 头号必考）**
> **中文**：**DQN=用神经网络近似 Q + 经验回放 + 目标网络**。**为什么需要这两件套**:RL 数据相关(非i.i.d.)+ 自举目标非平稳→直接训练发散。回放打破相关性&复用数据(靠 off-policy); 目标网络冻结目标稳定训练。损失=TD 误差的平方(用目标网络算目标)。**"致命三要素(deadly triad)"**:函数近似+自举+off-policy 三者同时出现易发散,DQN 的技巧就是缓解它。**重要变体**:Double DQN(解 max 过估计)、Dueling DQN(拆 V+Advantage)、Prioritized Replay(重要样本多采)、Rainbow(集大成)。适用**离散动作**;连续动作要 DDPG/SAC。
> **English**: **DQN = neural-net Q approximation + experience replay + target network**. **Why both are needed**: RL data is correlated (non-i.i.d.) + bootstrapped targets are non-stationary → naive training diverges. Replay breaks correlation & reuses data (thanks to off-policy); the target network freezes the target to stabilize. Loss = squared TD error (target computed with the target net). The **"deadly triad"**: function approximation + bootstrapping + off-policy together tend to diverge; DQN's tricks mitigate it. **Key variants**: Double DQN (fixes max overestimation), Dueling DQN (splits V + Advantage), Prioritized Replay (sample important transitions more), Rainbow (combines them). For **discrete actions**; continuous needs DDPG/SAC.


In [ ]:

# ============================================================
# 环境:从零实现 CartPole 物理 / CartPole physics from scratch
# 中文:一根杆子铰接在小车上, 车只能左右推。目标:通过左右推车让杆子不倒。
#      状态=(车位置, 车速度, 杆角度, 杆角速度) 4维连续。动作:0=左推 1=右推。
#      每存活一步 +1 奖励; 杆倒(>12°)或车出界(|x|>2.4)或到500步则结束。经典连续状态控制问题。
# English: a pole hinged on a cart; the cart can only be pushed left/right. Goal: keep the pole upright.
#      State=(cart x, cart velocity, pole angle, pole angular velocity), 4-dim continuous. Actions: 0=left,1=right.
#      +1 reward per surviving step; ends if pole falls (>12°), cart out of bounds (|x|>2.4), or 500 steps.
# ============================================================
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, random, time
from collections import deque
import matplotlib.pyplot as plt
def set_seed(x): torch.manual_seed(x); np.random.seed(x); random.seed(x)

class CartPole:
    g=9.8; mc=1.0; mp=0.1; l=0.5; fm=10.0; tau=0.02          # 重力/车质量/杆质量/半长/推力/时间步 / physics constants
    def reset(s): s.state=np.random.uniform(-0.05,0.05,4); s.steps=0; return s.state.copy()
    def step(s, a):
        x,xd,th,thd = s.state; force = s.fm if a==1 else -s.fm
        ct,st = np.cos(th), np.sin(th); tot=s.mc+s.mp
        temp=(force + s.mp*s.l*thd**2*st)/tot                # 中间量 / intermediate
        thacc=(s.g*st - ct*temp)/(s.l*(4/3 - s.mp*ct**2/tot))# 杆角加速度 / pole angular accel
        xacc = temp - s.mp*s.l*thacc*ct/tot                  # 车加速度 / cart accel
        x+=s.tau*xd; xd+=s.tau*xacc; th+=s.tau*thd; thd+=s.tau*thacc   # 欧拉积分一步 / Euler integration
        s.state=np.array([x,xd,th,thd]); s.steps+=1
        done = abs(x)>2.4 or abs(th)>12*np.pi/180 or s.steps>=500
        return s.state.copy(), 1.0, done                     # 每步 +1 / +1 per step
env=CartPole()
s=env.reset(); print("初始状态 / initial state:", np.round(s,3))
print("随机推右一步 / push right:", np.round(env.step(1)[0],3), "reward 1, done", env.step(1)[2])


**中文**：从零实现 DQN 的三大件:**Q 网络**(4维状态→2个动作的Q值)、**回放池**、**目标网络**,以及训练循环。为了**做消融实验**,我们把"是否用回放""是否用目标网络"做成开关——好直接看这两个技巧到底有多关键。
**English**: Implement DQN's three parts from scratch: the **Q-network** (4-dim state → Q-values for 2 actions), the **replay buffer**, and the **target network**, plus the training loop. To run **ablations**, we make "use replay" and "use target network" toggles — to directly see how crucial each trick is.


In [ ]:

# ============================================================
# 从零实现 DQN(带消融开关)/ DQN from scratch (with ablation switches)
# ============================================================
class QNet(nn.Module):
    def __init__(s): super().__init__(); s.f=nn.Sequential(nn.Linear(4,128),nn.ReLU(),nn.Linear(128,2))
    def forward(s,x): return s.f(x)                           # 输出两个动作的 Q 值 / Q-values for 2 actions

def train_dqn(episodes=350, use_replay=True, use_target=True, gamma=0.99, seed=0):
    set_seed(seed); env=CartPole()
    q=QNet(); tgt=QNet(); tgt.load_state_dict(q.state_dict()) # 主网络 + 目标网络 / online + target nets
    opt=torch.optim.Adam(q.parameters(), lr=1e-3)
    buf=deque(maxlen=20000); lengths=[]
    for ep in range(episodes):
        s=env.reset(); done=False; steps=0; eps=max(0.02, 1.0-ep/200)   # ε 线性衰减 / linear ε decay
        while not done:
            if random.random()<eps: a=random.randint(0,1)               # 探索 / explore
            else:
                with torch.no_grad(): a=int(q(torch.tensor(s,dtype=torch.float32)).argmax())  # 利用 / exploit
            ns,r,done=env.step(a); steps+=1
            buf.append((s,a,r,ns,done)); s=ns                            # 存入回放池 / store transition
            # 取训练批:回放=随机采样, 无回放=只用最新一条(暴露相关性问题)/ minibatch source
            if use_replay:
                if len(buf)<64: continue
                batch=random.sample(buf,64)
            else:
                batch=[buf[-1]]
            S =torch.tensor(np.array([b[0] for b in batch]),dtype=torch.float32)
            Aa=torch.tensor([b[1] for b in batch]); Rr=torch.tensor([b[2] for b in batch],dtype=torch.float32)
            NS=torch.tensor(np.array([b[3] for b in batch]),dtype=torch.float32); Dd=torch.tensor([b[4] for b in batch],dtype=torch.float32)
            qsa=q(S).gather(1,Aa.unsqueeze(1)).squeeze(1)               # Q(s,a) / predicted
            with torch.no_grad():
                net = tgt if use_target else q                          # 目标用冻结网络 or 主网络 / target source
                target=Rr + gamma*net(NS).max(1)[0]*(1-Dd)              # TD 目标 / TD target
            loss=F.smooth_l1_loss(qsa,target)                           # Huber 损失 / Huber loss
            opt.zero_grad(); loss.backward(); opt.step()
        lengths.append(steps)
        if use_target and ep%5==0: tgt.load_state_dict(q.state_dict())  # 定期同步目标网络 / sync target
    return lengths

t=time.time(); full=train_dqn(350, use_replay=True, use_target=True)
print(f"完整 DQN:最后50回合平均存活步数 {np.mean(full[-50:]):.0f} (最长 {max(full)}, 满分500) | {time.time()-t:.0f}s")
print("→ 从零(全随机)开始, DQN 学会了平衡杆子! / DQN learned to balance the pole from scratch!")


**中文**：完整 DQN 学会了平衡杆子(存活步数逼近满分 500)。现在做本节的**核心消融实验**——分别**关掉经验回放**和**关掉目标网络**,看会发生什么。这直接检验"这两个技巧到底是不是必需的"。
**English**: The full DQN learned to balance the pole (survival approaching the max of 500). Now the section's **core ablation** — turn off **experience replay** and **the target network** separately, and see what happens. A direct test of "are these two tricks really necessary?"


In [ ]:

# ============================================================
# 消融实验:关掉回放 / 关掉目标网络 / ablations
# ============================================================
t=time.time()
no_replay = train_dqn(350, use_replay=False, use_target=True)   # 只用最新一条转移(数据相关)/ no replay
no_target = train_dqn(350, use_replay=True,  use_target=False)  # 目标随主网络一起动(非平稳)/ no target net
print(f"{'配置/config':<26}{'最后50回合平均存活':>20}")
print(f"{'完整 DQN full':<26}{np.mean(full[-50:]):>20.0f}")
print(f"{'去掉经验回放 no-replay':<26}{np.mean(no_replay[-50:]):>20.0f}")
print(f"{'去掉目标网络 no-target':<26}{np.mean(no_target[-50:]):>20.0f}")
print(f"(随机策略基线 ~ 20 步 / random baseline ~20)  用时 {time.time()-t:.0f}s")


In [ ]:

# ============================================================
# 可视化 / Visualization
# ============================================================
def smooth(x,k=15): return np.convolve(x,np.ones(k)/k,mode="valid")
fig,ax=plt.subplots(1,2,figsize=(14,4.7))
# ① 完整 DQN 学习曲线 / full DQN learning curve
ax[0].plot(full,alpha=0.3,color="#4C72B0"); ax[0].plot(smooth(full),color="#C44E52",lw=2,label="15-ep 平滑")
ax[0].axhline(500,ls=":",color="gray",label="满分 max=500"); ax[0].axhline(20,ls="--",color="gray",label="随机 ~20")
ax[0].set_title("完整 DQN:从随机到会平衡 / full DQN learns"); ax[0].set_xlabel("episode"); ax[0].set_ylabel("存活步数 survival"); ax[0].legend(fontsize=8)
# ② 消融对比 / ablation comparison
ax[1].plot(smooth(full),color="#4C72B0",lw=2,label=f"完整 full ({np.mean(full[-50:]):.0f})")
ax[1].plot(smooth(no_replay),color="#C44E52",lw=2,label=f"无回放 no-replay ({np.mean(no_replay[-50:]):.0f})")
ax[1].plot(smooth(no_target),color="#DD8452",lw=2,label=f"无目标网 no-target ({np.mean(no_target[-50:]):.0f})")
ax[1].set_title("消融:两个技巧缺一不可 / both tricks are essential"); ax[1].set_xlabel("episode"); ax[1].set_ylabel("存活步数 survival"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig("/tmp/rl06_viz.png",dpi=80); plt.show()
print("去掉任一技巧, DQN 都学不起来(停在~10步) —— 这就是它们被发明的原因")
print("Remove either trick and DQN fails to learn (~10 steps) — exactly why they were invented")


**中文**：诚实解读:
**English**: Honest takeaways:

**中文**：
1. **DQN 从零学会了连续状态控制**:CartPole 的状态是 4 维**连续**的——表格法根本无从下手(无限多状态)。DQN 用一个小神经网络近似 Q,从全随机开始、只靠"每活一步+1"的奖励,几百回合就学会稳稳平衡杆子(逼近满分 500)。
2. **消融实验是本节的灵魂——两个技巧缺一不可**:去掉经验回放(平均~10步)或去掉目标网络(平均~10步),DQN 都**彻底学不起来**,甚至不如随机(~20步)!这不是调参问题,而是**结构性必需**:没有回放,相关的连续数据让梯度互相打架;没有目标网络,你追一个随自己移动的目标,直接发散。**这正是 2015 年 DQN 论文的核心贡献**——不是"用神经网络做 Q-learning"(前人试过、都失败了),而是"用这两个技巧让它**终于稳定收敛**"。
3. **诚实的稳定性说明**:DQN 训练**天生不稳定**(受随机种子影响大,曲线会剧烈波动、甚至"学会又忘掉"catastrophic forgetting)。你重跑可能看到不同曲线——这是深度 RL 的常态,也是为什么后来有一堆稳定化改进(Double/Dueling/PER/Rainbow)。

**English**:
1. **DQN learned continuous-state control from scratch**: CartPole's state is 4-dim **continuous** — tables are hopeless (infinitely many states). DQN approximates Q with a small network and, from fully random with only "+1 per surviving step," learns in a few hundred episodes to balance the pole steadily (approaching the max of 500).
2. **The ablation is the soul of this section — both tricks are indispensable**: removing experience replay (~10 steps avg) or the target network (~10 steps avg) makes DQN **completely fail to learn**, worse even than random (~20)! Not a tuning issue but a **structural necessity**: without replay, correlated consecutive data makes gradients fight; without a target network, you chase a target that moves with you and diverge. **This is the core contribution of the 2015 DQN paper** — not "do Q-learning with a neural net" (predecessors tried and failed) but "make it **finally converge stably** with these two tricks."
3. **An honest stability note**: DQN training is **inherently unstable** (very seed-sensitive; curves swing wildly and can even "learn then forget" — catastrophic forgetting). Re-running may show a different curve — this is normal in deep RL and why later work piled on stabilizers (Double/Dueling/PER/Rainbow).

> 💼 **实战视角 / Practical angle**
> **中文**:DQN 是深度 RL 的**开山之作**(2015 Atari 从像素到人类水平),适用**离散动作**(游戏、推荐排序、资源调度)。**工程要点**:①回放池大小、目标网络同步频率、ε 衰减都很敏感;②几乎必上 **Double DQN**(缓解 max 过估计)+ **Dueling** + **优先回放**;③连续动作(机器人力矩)不能用 DQN(argmax 无法枚举)→ 用 **DDPG/TD3/SAC**;④真实系统里 DQN 样本效率低、不稳定,常被 PPO(下几节)或离线 RL 取代。面试金句:*"DQN=神经网络近似 Q + 经验回放(破相关、复用数据)+ 目标网络(稳目标); 没这两件套会因'致命三要素'发散——消融一关就崩,这就是它们存在的意义。"*
> **English**: DQN is deep RL's **founding work** (2015 Atari, pixels to human level), for **discrete actions** (games, ranking, resource scheduling). **Engineering**: ① replay size, target-sync frequency, and ε decay are all sensitive; ② almost always add **Double DQN** (mitigate max overestimation) + **Dueling** + **Prioritized Replay**; ③ continuous actions (robot torques) can't use DQN (can't argmax over infinite actions) → use **DDPG/TD3/SAC**; ④ in real systems DQN is sample-inefficient and unstable, often replaced by PPO (coming sections) or offline RL. Interview line: *"DQN = neural-net Q approximation + experience replay (break correlation, reuse data) + target network (stabilize the target); without these two it diverges from the 'deadly triad' — ablate either and it collapses, which is exactly why they exist."*

---
### 小结 / Summary
- **中文**:DQN=用神经网络近似 Q, 解决表格法撑不起大/连续状态空间的问题(Atari 从像素到人类水平)。
- **English**: DQN = neural-net Q approximation, solving tables' failure on large/continuous state spaces (Atari, pixels to human level).
- **中文**:两大稳定化技巧缺一不可——经验回放(破相关+复用)、目标网络(稳目标); 消融一关就崩。
- **English**: Two stabilizers are indispensable — experience replay (break correlation + reuse) and target network (stable target); ablate either and it collapses.
- **中文**:DQN 只适用离散动作、训练不稳、样本效率低——引出策略梯度/Actor-Critic/PPO 与连续控制。
- **English**: DQN is discrete-action-only, unstable, sample-inefficient — motivating policy gradients / Actor-Critic / PPO and continuous control.
